## Deep-kernel MMD two-sample test: which model's samples are closest to real music (Kaggle-ready, self-contained)

Implements `docs/mmd-two-sample-test.md`: the learned deep-kernel MMD test (MMD-D, Liu et al. 2020, Algorithm 1) on CLaMP 2 embeddings, comparing each model's generated 8-bar excerpts with held-out real excerpts.

Pipeline: draw real chunks A (val split, conditions + control) and B (test split, reference) → each model generates one excerpt per A attribute grid → tokens → MIDI → CLaMP 2 embedding → for 10 random splits: train a deep kernel on fit/val, permutation-test on the held-out half, for `plain vs B`, `simple vs B`, `hier vs B`, and the real-vs-real control `A vs B`, plus a cross-kernel control per model.

### Run modes
- **Dry run (about 15 min):** set `MODE = 'dry'` in the first code cell (Kaggle cannot pass CLI flags). Outside Kaggle, `--dry-run` on the command line or `DRY_RUN=1` also works.
- **Full run (fits the 9 h session):** `MODE = 'full'`, then **Save Version → Save & Run All**. A timing pilot picks N (up to 5000 per model) to fit the budget; a deadline guard writes results before the session ends.

### Kaggle settings
- Accelerator: **GPU T4 x2** (one GPU is used). **Internet: on** (pip install + CLaMP 2 weights, about 5.5 GB from Hugging Face).
- Datasets under `/kaggle/input/datasets/karandeepshoker/<slug>/`: `plain_v3/ckpt.pt`, `simple_v2/ckpt.pt`, `vae_v3/ckpt.pt`, `preprocessed-memmapval/`, `preprocessed-memmaptest/`, `misc/` (`tokenizer_meta.json`, `quantiles.json`).

### Outputs (`/kaggle/working/artifacts/mmd_test/`)
`results.json` (written after every split), `summary.md`, `tokens.npz`, `emb_<set>.npy`, `midi_samples/<set>/`.

In [ ]:
from __future__ import annotations

import os
import sys
import json
import math
import time
import shutil
import subprocess
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

T0 = time.time()

# -------------------------
# Run mode
#   'dry'  : small end-to-end test (about 15 min on Kaggle)
#   'full' : overnight run, fits the 9 h session
#   None   : read --dry-run from the command line or DRY_RUN=1 from the environment (default: full)
# -------------------------
MODE = None

if MODE is None:
    DRY_RUN = ('--dry-run' in sys.argv) or (os.environ.get('DRY_RUN', '') == '1')
else:
    assert MODE in ('dry', 'full'), MODE
    DRY_RUN = MODE == 'dry'

print('=' * 64)
print('MODE:', 'DRY RUN (small end-to-end test)' if DRY_RUN else 'FULL RUN')
print('=' * 64)

print('cwd:', os.getcwd())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), '| count:', torch.cuda.device_count())

INPUT_ROOT = Path(os.environ.get('MMD_INPUT_ROOT', '/kaggle/input'))
WORKING_ROOT = Path(os.environ.get('MMD_WORKING_ROOT', '/kaggle/working'))
OUT_DIR = WORKING_ROOT / 'artifacts' / ('mmd_test_dry' if DRY_RUN else 'mmd_test')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if INPUT_ROOT.exists():
    print('--- input ---')
    for p in sorted(INPUT_ROOT.rglob('*')):
        if p.is_dir() and len(p.relative_to(INPUT_ROOT).parts) <= 3:
            print(' -', p.relative_to(INPUT_ROOT))


def elapsed() -> float:
    return time.time() - T0


def fmt_secs(s: float) -> str:
    s = max(0.0, float(s))
    if s < 60:
        return f'{s:5.1f}s'
    if s < 3600:
        return f'{s / 60:5.1f}m'
    return f'{s / 3600:5.2f}h'

In [ ]:
# -------------------------
# Config (see docs/mmd-two-sample-test.md)
# -------------------------

KAGGLE_USER = 'karandeepshoker'

RUNS = {'plain': 'plain_v3', 'simple_vae': 'simple_v2', 'vae': 'vae_v3'}
LABEL = {'plain': 'plain', 'simple_vae': 'simple', 'vae': 'hier'}
MODEL_TYPES = ['plain', 'simple_vae', 'vae']

# Generated sets: label -> model type. 'plain2' is a second, independently seeded plain sample on the
# same conditions; plain vs plain2 calibrates the relative-similarity test (true difference is zero).
GEN_SPECS = {'plain': 'plain', 'simple': 'simple_vae', 'hier': 'vae', 'plain2': 'plain'}
MODEL_LABELS = ['plain', 'simple', 'hier']
REL_PAIRS = [('plain', 'simple'), ('plain', 'hier'), ('simple', 'hier'), ('plain', 'plain2')]

VAL_DATASET = 'preprocessed-memmapval'
TEST_DATASET = 'preprocessed-memmaptest'
MISC_DATASET = 'misc'
TOKENIZER_META_NAME = 'tokenizer_meta.json'
QUANTILES_NAME = 'quantiles.json'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 0

# Sampling: same as the listening test / demo. Generation starts from a Bar token at position 0, as
# every training chunk does (BOS never occurs in the training memmaps).
TEMPERATURE = 1.0
TOP_P = 0.9
BLOCK_SIZE = 1024
BARS = 8

CFG = dict(
    N_MAX=10000,           # excerpts per model (upper bound; the pilot may lower it)
    N_FLOOR=1000,          # warn if the time budget forces N below this
    GEN_BATCH=128,
    N_SPLITS=10,
    FRAC_TEST=0.5,
    FRAC_VAL=0.1,          # of each side; the rest is the fit set
    PHI_WIDTH=256,         # hidden and output width of phi
    LR=1e-4,               # paper range 1e-5 (HDGM) to 1e-3; 1e-4 won a synthetic pilot at this n
    STEPS=2000,            # upper bound; patience stops training once validation J stalls
    VAL_EVERY=10,          # best validation J lands early (about step 30-40 in the pilot)
    EARLY_STOP=True,       # set False to train a fixed STEPS (if val and held-out J diverge in the dry run)
    PATIENCE=300,          # stop when val J has not improved for this many steps
    N_PERM=200,
    ALPHA=0.05,
    KV_CHECK_TOKENS=128,
    KV_CHECK_TOL=1e-3,
    N_MIDI_SAMPLES=10,
    EMB_PILOT_FILES=24,
    BLOCK_WIDTHS=(200, 1000),  # memmap-index blocks for cluster-robust SEs (pieces are stored contiguously);
                               # decisions use the more conservative of the two
    SPLIT_BLOCK_WIDTH=500,     # fit/val/test are split by these blocks, so no piece sits on both sides of a split
    # Time budget, seconds. The session limit is 9 h; everything is planned to finish by DEADLINE_S.
    DEADLINE_S=8.25 * 3600,
    GEN_BUDGET_S=2.5 * 3600,
    EMB_BUDGET_S=2.5 * 3600,
    TEST_RESERVE_S=3.0 * 3600,
)

if DRY_RUN:
    CFG.update(
        N_MAX=64, N_FLOOR=16, GEN_BATCH=32,
        N_SPLITS=1, STEPS=50, VAL_EVERY=5, PATIENCE=50, N_PERM=50,
        KV_CHECK_TOKENS=48, N_MIDI_SAMPLES=3, EMB_PILOT_FILES=8,
        DEADLINE_S=0.5 * 3600, GEN_BUDGET_S=6 * 60, EMB_BUDGET_S=8 * 60, TEST_RESERVE_S=4 * 60,
    )

print(json.dumps(CFG, indent=1))
print('OUT_DIR:', OUT_DIR)
print('DEVICE:', DEVICE)

In [ ]:
# -------------------------
# Install CLaMP 2 feature extractor (frechet-music-distance) + mido
# --no-deps first so Kaggle's torch/transformers are not touched; full install as fallback.
# -------------------------

def _pip(*args) -> int:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
    return r.returncode


try:
    import mido
    from frechet_music_distance.models import CLaMP2Extractor
except Exception:
    _pip('--no-deps', 'frechet-music-distance')
    _pip('mido', 'abctoolkit', 'unidecode')
    try:
        import mido
        from frechet_music_distance.models import CLaMP2Extractor
    except Exception as e:
        print('minimal install failed, installing with dependencies:', repr(e))
        _pip('frechet-music-distance', 'mido')
        import mido
        from frechet_music_distance.models import CLaMP2Extractor

print('mido + CLaMP2Extractor import ok')

In [ ]:
# -------------------------
# Locate dataset paths (same layout as notebooks 04/05/08)
# -------------------------

def require_dir(p: Path) -> Path:
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(f'missing directory: {p}')
    return p


def require_file(p: Path) -> Path:
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f'missing file: {p}')
    return p


def find_split_dir(root: Path, rel_candidates) -> Path:
    for rel in rel_candidates:
        cand = root / rel
        if cand.exists() and (cand / 'meta.json').exists():
            for f in ['meta.json', 'X.u16.memmap', 'bar_indices.u8.memmap', 'attributes.u8.memmap']:
                require_file(cand / f)
            return cand
    raise FileNotFoundError(f'Could not find split in {root} (tried {rel_candidates})')


BASE = require_dir(INPUT_ROOT / 'datasets' / KAGGLE_USER)

CKPT = {mt: require_file(require_dir(BASE / run) / 'ckpt.pt') for mt, run in RUNS.items()}

MISC_DIR = require_dir(BASE / MISC_DATASET)
TOKENIZER_META_PATH = require_file(MISC_DIR / TOKENIZER_META_NAME)
QUANTILES_PATH = require_file(MISC_DIR / QUANTILES_NAME)

VAL_DIR = find_split_dir(require_dir(BASE / VAL_DATASET), [Path('val'), Path('.')])
TEST_DIR = find_split_dir(require_dir(BASE / TEST_DATASET), [Path('test'), Path('.')])

for mt, p in CKPT.items():
    print(f'{LABEL[mt]:>6} ckpt:', p)
print('VAL_DIR:', VAL_DIR)
print('TEST_DIR:', TEST_DIR)
print('TOKENIZER_META_PATH:', TOKENIZER_META_PATH)
print('QUANTILES_PATH:', QUANTILES_PATH)

In [ ]:
# -------------------------
# Self-contained code (write a tiny module file) - same pattern as 04/05/08
# Model, grammar and attribute code: verbatim from notebook 08 (repo code).
# MMD_CODE: new for this notebook (KV-cached batched generation, REMI->MIDI, deep-kernel MMD).
# -------------------------

import importlib

MODULE_PATH = Path('musicgen_kaggle_mmd_min.py')

PLAIN_MODEL_CODE = r'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class PlainTransformerConfig:
    vocab_size: int = 195
    block_size: int = 1024
    n_layers: int = 6
    n_embed: int = 512
    n_head: int = 8
    d_ff: int = 2048
    bars_per_sample: int = 8
    dropout: float = 0.1

    # Attributes: 4 dims (polyphony, intensity, velocity, density), each binned 0..7
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128


class AttributeConditioner(nn.Module):
    """
    attributes: LongTensor [B, bars, 4] with values in 0..7
    returns:    FloatTensor [B, bars, n_embed]
    """

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        assert cfg.attribute_embed_dim * cfg.num_attributes == 128

        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )
        self.proj = nn.Linear(cfg.attribute_embed_dim * cfg.num_attributes, cfg.n_embed)

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        # attributes: [B, bars, 4]
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        A = torch.cat([a0, a1, a2, a3], dim=-1)  # [B, bars, 128]
        return self.proj(A)  # [B, bars, n_embed]


class PlainCausalSelfAttention(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.n_embed % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embed // cfg.n_head

        self.qkv = nn.Linear(cfg.n_embed, 3 * cfg.n_embed, bias=True)
        self.proj = nn.Linear(cfg.n_embed, cfg.n_embed, bias=True)
        self.dropout = float(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, C]
        B, T, C = x.shape
        qkv = self.qkv(x)  # [B, T, 3C]
        q, k, v = qkv.split(C, dim=-1)

        # [B, nh, T, hs]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # PyTorch uses optimized SDPA under the hood when available.
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )  # [B, nh, T, hs]
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # [B, T, C]
        return self.proj(y)


class PlainFiLMBlock(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embed)
        self.attn = PlainCausalSelfAttention(cfg)
        self.resid_drop = nn.Dropout(cfg.dropout)

        # Conditioned FiLM: gamma/beta per token timestep, derived from cond_tok [B,T,C]
        self.film = nn.Linear(cfg.n_embed, 2 * cfg.n_embed, bias=True)

        self.ln2 = nn.LayerNorm(cfg.n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embed, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.n_embed),
        )
        self.mlp_drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        # Pre-norm
        h = self.ln1(x)
        gamma_beta = self.film(cond_tok)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))

        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class PlainTransformerDecoder(nn.Module):
    """Decoder-only baseline conditioned ONLY on per-bar attributes via FiLM."""

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embed)

        self.attr = AttributeConditioner(cfg)
        self.blocks = nn.ModuleList([PlainFiLMBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.n_embed)
        self.head = nn.Linear(cfg.n_embed, cfg.vocab_size, bias=True)

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
    ) -> dict:
        B, T = X.shape
        if T > self.cfg.block_size:
            raise ValueError(f"Sequence length T={T} exceeds block_size={self.cfg.block_size}")

        # Token + position embeddings
        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]

        # Build per-bar conditions and broadcast to tokens using bar_indices
        cond_bars = self.attr(attributes)  # [B, bars, C]
        idx = bar_indices.to(dtype=torch.long).clamp_(0, self.cfg.bars_per_sample - 1)
        cond_tok = cond_bars.gather(1, idx.unsqueeze(-1).expand(-1, -1, self.cfg.n_embed))  # [B, T, C]

        for blk in self.blocks:
            x = blk(x, cond_tok)

        x = self.ln_f(x)
        logits = self.head(x)  # [B, T, vocab]

        out = {"logits": logits}
        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing loss with targets.")

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            n_tokens = int((targets != int(pad_id)).sum().item())
            out.update({"loss": loss, "n_tokens": n_tokens})

        return out
'''

SIMPLE_MODEL_CODE = r'''
from dataclasses import dataclass
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class SimpleVAEConfig:
    vocab_size: int = 195
    block_size: int = 1024

    # Encoder (bidirectional)
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1

    # Attributes
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128

    # Simple global latent projection (no Conductor)
    z_global_dim: int = 384

    # Decoder (causal)
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1


class AttributeEmbedder(nn.Module):
    """
    attributes: LongTensor [B, bars, 4] with values 0..7
    returns:    FloatTensor [B, bars, 128]
    """

    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        return torch.cat([a0, a1, a2, a3], dim=-1)  # [B, bars, 128]


def broadcast_by_bar_indices(bars_x_c: torch.Tensor, bar_indices: torch.Tensor) -> torch.Tensor:
    """bars_x_c: [B,bars,C], bar_indices: [B,T] -> [B,T,C]"""
    B, bars, C = bars_x_c.shape
    idx = bar_indices.to(dtype=torch.long).clamp_(0, bars - 1)
    return bars_x_c.gather(1, idx.unsqueeze(-1).expand(B, idx.size(1), C))


class BiTransformerEncoder(nn.Module):
    """Bidirectional encoder with learned [CLS] token, outputs mu/logvar for z_p."""

    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)  # +1 for CLS position
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.enc_n_embed,
            nhead=cfg.enc_n_head,
            dim_feedforward=cfg.enc_d_ff,
            dropout=float(cfg.enc_dropout),
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.ln_f = nn.LayerNorm(cfg.enc_n_embed)

        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)

        nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Encoder expects block_size={self.cfg.block_size}, got T={T}")

        tok = self.tok_emb(X)  # [B,T,C]
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)  # [B,1,C]
        h = torch.cat([cls, tok], dim=1)  # [B,T+1,C]

        pos = torch.arange(T + 1, device=X.device)
        h = h + self.pos_emb(pos)[None, :, :]

        h = self.encoder(h)
        h = self.ln_f(h)
        h_cls = h[:, 0, :]  # [B,C]
        return self.mu(h_cls), self.logvar(h_cls)


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed: int, n_head: int):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=True)
        self.proj = nn.Linear(n_embed, n_embed, bias=True)
        self.dropout = 0.0

    def set_dropout(self, p: float) -> None:
        self.dropout = float(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class FiLMBlock(nn.Module):
    def __init__(self, n_embed: int, n_head: int, d_ff: int, *, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed=n_embed, n_head=n_head)
        self.attn.set_dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.film = nn.Linear(n_embed, 2 * n_embed, bias=True)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, n_embed),
        )
        self.mlp_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class FiLMDecoder(nn.Module):
    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)

        self.blocks = nn.ModuleList(
            [
                FiLMBlock(cfg.dec_n_embed, cfg.dec_n_head, cfg.dec_d_ff, dropout=float(cfg.dec_dropout))
                for _ in range(cfg.dec_n_layers)
            ]
        )
        self.ln_f = nn.LayerNorm(cfg.dec_n_embed)
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size, bias=True)

    def forward(self, X: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Decoder expects block_size={self.cfg.block_size}, got T={T}")

        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        return self.head(x)  # [B,T,vocab]


class SimpleMusicVAE(nn.Module):
    """Simple (non-hierarchical) conditional VAE."""

    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        self.cfg = cfg
        self.attr = AttributeEmbedder(cfg)  # [B,8,128]
        self.encoder = BiTransformerEncoder(cfg)
        self.decoder = FiLMDecoder(cfg)

        self.z_proj = nn.Linear(cfg.z_dim, cfg.z_global_dim)

        cond_dim = int(cfg.z_global_dim) + int(cfg.attribute_embed_dim * cfg.num_attributes)
        if cond_dim != cfg.dec_n_embed:
            raise ValueError(f"Expected z_global_dim+attr_dim == dec_n_embed; got {cond_dim} vs {cfg.dec_n_embed}")

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    @staticmethod
    def kl_divergence_free_bits(
        mu: torch.Tensor,
        logvar: torch.Tensor,
        *,
        free_bits_lambda_bits: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        kl_per_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
        kl_per_dim_mean = kl_per_dim.mean(dim=0)
        lambda_nats = float(free_bits_lambda_bits) * math.log(2.0)
        kl_per_dim_hinged_mean = torch.clamp(kl_per_dim_mean, min=lambda_nats)
        kl_loss = kl_per_dim_hinged_mean.sum()
        active_units = (kl_per_dim_mean > lambda_nats).sum()
        return kl_loss, kl_per_dim_mean, active_units

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        z_p: Optional[torch.Tensor] = None,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
        beta: float = 1.0,
    ) -> dict:
        mu: Optional[torch.Tensor]
        logvar: Optional[torch.Tensor]
        if z_p is None:
            mu, logvar = self.encoder(X)
            z_p = self.reparameterize(mu, logvar)
        else:
            mu, logvar = None, None
            if z_p.ndim != 2 or z_p.shape[0] != X.shape[0]:
                raise ValueError(f"z_p must have shape [B,z_dim], got {tuple(z_p.shape)}")
            if z_p.shape[1] != self.cfg.z_dim:
                raise ValueError(f"z_p dim mismatch: expected {self.cfg.z_dim}, got {z_p.shape[1]}")

        z_g = self.z_proj(z_p)
        z_g_bars = z_g[:, None, :].expand(-1, self.cfg.bars_per_sample, -1)

        A_k = self.attr(attributes)
        C_k = torch.cat([z_g_bars, A_k], dim=-1)
        cond_tok = broadcast_by_bar_indices(C_k, bar_indices)

        logits = self.decoder(X, cond_tok)

        out = {"logits": logits, "mu": mu, "logvar": logvar, "z_p": z_p, "z_g": z_g}

        if mu is not None and logvar is not None:
            loss_kl, kl_per_dim_mean, active_units = self.kl_divergence_free_bits(mu, logvar, free_bits_lambda_bits=1.0)
            out["loss_kl"] = loss_kl
            out["kl_per_dim_mean"] = kl_per_dim_mean
            out["active_units"] = active_units

        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing reconstruction loss.")
            if mu is None or logvar is None:
                raise ValueError("targets provided but z_p was passed explicitly; KL stats are undefined in this mode.")
            loss_recon = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            out["loss_recon"] = loss_recon
            out["loss_total"] = loss_recon + float(beta) * loss_kl

        return out
'''

VAE_MODEL_CODE = r'''
from dataclasses import dataclass
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class VAEConfig:
    vocab_size: int = 195
    block_size: int = 1024

    # Encoder (bidirectional)
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1

    # Attributes
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128

    # Conductor
    cond_hidden: int = 384
    cond_layers: int = 2

    # Decoder (causal)
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1


class AttributeEmbedder(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        return torch.cat([a0, a1, a2, a3], dim=-1)


def broadcast_by_bar_indices(bars_x_c: torch.Tensor, bar_indices: torch.Tensor) -> torch.Tensor:
    B, bars, C = bars_x_c.shape
    idx = bar_indices.to(dtype=torch.long).clamp_(0, bars - 1)
    return bars_x_c.gather(1, idx.unsqueeze(-1).expand(B, idx.size(1), C))


class BiTransformerEncoder(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.enc_n_embed,
            nhead=cfg.enc_n_head,
            dim_feedforward=cfg.enc_d_ff,
            dropout=float(cfg.enc_dropout),
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.ln_f = nn.LayerNorm(cfg.enc_n_embed)

        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)

        nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Encoder expects block_size={self.cfg.block_size}, got T={T}")

        tok = self.tok_emb(X)
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)
        h = torch.cat([cls, tok], dim=1)

        pos = torch.arange(T + 1, device=X.device)
        h = h + self.pos_emb(pos)[None, :, :]

        h = self.encoder(h)
        h = self.ln_f(h)
        h_cls = h[:, 0, :]
        return self.mu(h_cls), self.logvar(h_cls)


class Conductor(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.h0_proj = nn.Linear(cfg.z_dim, cfg.cond_layers * cfg.cond_hidden)
        self.bar_emb = nn.Embedding(cfg.bars_per_sample, cfg.cond_hidden)
        self.gru = nn.GRU(
            input_size=cfg.cond_hidden,
            hidden_size=cfg.cond_hidden,
            num_layers=cfg.cond_layers,
            batch_first=True,
        )

    def forward(self, z_p: torch.Tensor) -> torch.Tensor:
        B = z_p.size(0)
        h0 = self.h0_proj(z_p).view(self.cfg.cond_layers, B, self.cfg.cond_hidden).contiguous()

        bar_ids = torch.arange(self.cfg.bars_per_sample, device=z_p.device)
        x = self.bar_emb(bar_ids)[None, :, :].expand(B, -1, -1)

        z_seq, _hn = self.gru(x, h0)
        return z_seq


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed: int, n_head: int):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=True)
        self.proj = nn.Linear(n_embed, n_embed, bias=True)
        self.dropout = 0.0

    def set_dropout(self, p: float) -> None:
        self.dropout = float(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class FiLMBlock(nn.Module):
    def __init__(self, n_embed: int, n_head: int, d_ff: int, *, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed=n_embed, n_head=n_head)
        self.attn.set_dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.film = nn.Linear(n_embed, 2 * n_embed, bias=True)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, n_embed),
        )
        self.mlp_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class FiLMDecoder(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)

        self.blocks = nn.ModuleList(
            [
                FiLMBlock(cfg.dec_n_embed, cfg.dec_n_head, cfg.dec_d_ff, dropout=float(cfg.dec_dropout))
                for _ in range(cfg.dec_n_layers)
            ]
        )
        self.ln_f = nn.LayerNorm(cfg.dec_n_embed)
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size, bias=True)

    def forward(self, X: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Decoder expects block_size={self.cfg.block_size}, got T={T}")

        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        return self.head(x)


class MusicVAE(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.attr = AttributeEmbedder(cfg)
        self.encoder = BiTransformerEncoder(cfg)
        self.conductor = Conductor(cfg)
        self.decoder = FiLMDecoder(cfg)

        if cfg.cond_hidden + (cfg.attribute_embed_dim * cfg.num_attributes) != cfg.dec_n_embed:
            raise ValueError(
                f"Expected cond_hidden(={cfg.cond_hidden}) + attr_dim(={cfg.attribute_embed_dim*cfg.num_attributes}) == dec_n_embed(={cfg.dec_n_embed})"
            )

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    @staticmethod
    def kl_divergence_free_bits(
        mu: torch.Tensor,
        logvar: torch.Tensor,
        *,
        free_bits_lambda_bits: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        kl_per_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
        kl_per_dim_mean = kl_per_dim.mean(dim=0)
        lambda_nats = float(free_bits_lambda_bits) * math.log(2.0)
        kl_per_dim_hinged_mean = torch.clamp(kl_per_dim_mean, min=lambda_nats)
        kl_loss = kl_per_dim_hinged_mean.sum()
        active_units = (kl_per_dim_mean > lambda_nats).sum()
        return kl_loss, kl_per_dim_mean, active_units

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        z_p: Optional[torch.Tensor] = None,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
        beta: float = 1.0,
    ) -> dict:
        mu: Optional[torch.Tensor]
        logvar: Optional[torch.Tensor]
        if z_p is None:
            mu, logvar = self.encoder(X)
            z_p = self.reparameterize(mu, logvar)
        else:
            mu, logvar = None, None
            if z_p.ndim != 2 or z_p.shape[0] != X.shape[0]:
                raise ValueError(f"z_p must have shape [B,z_dim], got {tuple(z_p.shape)}")
            if z_p.shape[1] != self.cfg.z_dim:
                raise ValueError(f"z_p dim mismatch: expected {self.cfg.z_dim}, got {z_p.shape[1]}")

        z_k = self.conductor(z_p)
        A_k = self.attr(attributes)
        C_k = torch.cat([z_k, A_k], dim=-1)
        cond_tok = broadcast_by_bar_indices(C_k, bar_indices)

        logits = self.decoder(X, cond_tok)
        out = {"logits": logits, "mu": mu, "logvar": logvar, "z_p": z_p, "z_k": z_k}

        if mu is not None and logvar is not None:
            loss_kl, kl_per_dim_mean, active_units = self.kl_divergence_free_bits(mu, logvar, free_bits_lambda_bits=1.0)
            out["loss_kl"] = loss_kl
            out["kl_per_dim_mean"] = kl_per_dim_mean
            out["active_units"] = active_units

        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing reconstruction loss.")
            if mu is None or logvar is None:
                raise ValueError("targets provided but z_p was passed explicitly; KL stats are undefined in this mode.")
            loss_recon = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            out["loss_recon"] = loss_recon
            out["loss_total"] = loss_recon + float(beta) * loss_kl

        return out
'''

GRAMMAR_CODE = r'''
from dataclasses import dataclass
from typing import Iterable, Literal

import torch


TokenType = Literal[
    'PAD',
    'BOS',
    'Bar',
    'Position',
    'Tempo',
    'Pitch',
    'Velocity',
    'Duration',
    'Other',
]


def token_type(token_str: str) -> TokenType:
    head = token_str.split('_', 1)[0]
    if head == 'PAD':
        return 'PAD'
    if head == 'BOS':
        return 'BOS'
    if head == 'Bar':
        return 'Bar'
    if head == 'Position':
        return 'Position'
    if head == 'Tempo':
        return 'Tempo'
    if head == 'Pitch':
        return 'Pitch'
    if head == 'Velocity':
        return 'Velocity'
    if head == 'Duration':
        return 'Duration'
    return 'Other'


@dataclass(frozen=True)
class GrammarVocabIndex:
    ids_by_type: dict[TokenType, list[int]]


def build_vocab_index(id_to_token: dict[int, str]) -> GrammarVocabIndex:
    ids_by_type: dict[TokenType, list[int]] = {
        'PAD': [],
        'BOS': [],
        'Bar': [],
        'Position': [],
        'Tempo': [],
        'Pitch': [],
        'Velocity': [],
        'Duration': [],
        'Other': [],
    }
    for tid, s in id_to_token.items():
        ids_by_type[token_type(s)].append(int(tid))
    for k in ids_by_type:
        ids_by_type[k] = sorted(ids_by_type[k])
    return GrammarVocabIndex(ids_by_type=ids_by_type)


def allowed_next_ids(
    prev_token_id: int,
    *,
    id_to_token: dict[int, str],
    vocab_index: GrammarVocabIndex,
    bar_id: int,
    pad_id: int,
    bos_id: int,
    allow_bar: bool = True,
) -> list[int]:
    prev_str = id_to_token[int(prev_token_id)]
    prev_type = token_type(prev_str)

    if prev_token_id == int(bos_id) or prev_type == 'BOS':
        return [int(bar_id)]

    if prev_token_id == int(bar_id) or prev_type == 'Bar':
        return vocab_index.ids_by_type['Position']

    if prev_type == 'Pitch':
        return vocab_index.ids_by_type['Velocity']

    if prev_type == 'Velocity':
        return vocab_index.ids_by_type['Duration']

    if prev_type == 'Tempo':
        return vocab_index.ids_by_type['Position']

    if prev_type == 'Position':
        allowed: list[int] = []
        allowed += vocab_index.ids_by_type['Pitch']
        allowed += vocab_index.ids_by_type['Tempo']
        if allow_bar:
            allowed.append(int(bar_id))
        return sorted(set(allowed))

    if prev_type == 'Duration':
        allowed = []
        allowed += vocab_index.ids_by_type['Pitch']
        allowed += vocab_index.ids_by_type['Position']
        allowed += vocab_index.ids_by_type['Tempo']
        if allow_bar:
            allowed.append(int(bar_id))
        return sorted(set(allowed))

    allowed = []
    allowed += vocab_index.ids_by_type['Bar'] if allow_bar else []
    allowed += vocab_index.ids_by_type['Position']
    allowed += vocab_index.ids_by_type['Tempo']
    allowed += vocab_index.ids_by_type['Pitch']
    allowed = [i for i in allowed if i not in (int(pad_id), int(bos_id))]
    return sorted(set(allowed))


def mask_logits_inplace(
    logits: torch.Tensor,
    *,
    allowed_ids: Iterable[int],
) -> torch.Tensor:
    if logits.ndim != 1:
        raise ValueError(f'Expected logits [vocab], got {tuple(logits.shape)}')
    allowed = torch.tensor(list(allowed_ids), device=logits.device, dtype=torch.long)
    mask = torch.ones((logits.size(0),), device=logits.device, dtype=torch.bool)
    mask[allowed] = False
    logits[mask] = float('-inf')
    return logits
'''

ATTR_CODE = r'''
import json
from pathlib import Path

import numpy as np

ATTRIBUTE_NAMES = ['polyphony_rate', 'rhythmic_intensity', 'velocity_dynamics', 'note_density']

BLOCK_SIZE = 1024
BARS_PER_SAMPLE = 8


def _parse_duration_beats(token: str) -> float:
    raw = token.split('_', 1)[1]
    try:
        return float(raw)
    except Exception:
        parts = raw.split('.')
        if len(parts) == 3:
            a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
            if c == 0:
                raise ValueError(f'Cannot parse duration token: {token}')
            return float(a + (b / c))
        raise ValueError(f'Cannot parse duration token: {token}')


def compute_raw_attributes_per_bar_from_tokens(
    x_tokens_1024: list[int],
    *,
    id_to_token: dict[int, str],
    bar_token_id: int,
    bars_per_sample: int = BARS_PER_SAMPLE,
) -> list[list[float]]:
    if len(x_tokens_1024) != BLOCK_SIZE:
        raise ValueError(f'Expected {BLOCK_SIZE} tokens, got {len(x_tokens_1024)}')

    onsets = [0 for _ in range(bars_per_sample)]
    pos_has_onset: list[set[int]] = [set() for _ in range(bars_per_sample)]
    velocities: list[list[int]] = [[] for _ in range(bars_per_sample)]
    global_intervals: list[tuple[float, float]] = []

    cur_bar = -1
    cur_pos = 0

    def parse_pos(tok: str) -> int:
        return int(tok.split('_', 1)[1])

    def parse_vel(tok: str) -> int:
        return int(tok.split('_', 1)[1])

    i = 0
    while i < len(x_tokens_1024):
        tid = x_tokens_1024[i]
        tok = id_to_token.get(int(tid))
        if tok is None:
            i += 1
            continue
        typ = tok.split('_', 1)[0]

        if tid == bar_token_id or typ == 'Bar':
            cur_bar += 1
            cur_pos = 0
            i += 1
            continue

        if cur_bar < 0:
            i += 1
            continue
        if cur_bar >= bars_per_sample:
            break

        if typ == 'Position':
            cur_pos = parse_pos(tok)
            i += 1
            continue

        if typ == 'Pitch' and i + 2 < len(x_tokens_1024):
            vtok = id_to_token.get(int(x_tokens_1024[i + 1]), '')
            dtok = id_to_token.get(int(x_tokens_1024[i + 2]), '')
            if vtok.startswith('Velocity_') and dtok.startswith('Duration_'):
                vel = parse_vel(vtok)
                dur = _parse_duration_beats(dtok)

                onsets[cur_bar] += 1
                pos_has_onset[cur_bar].add(cur_pos)
                velocities[cur_bar].append(vel)

                s = cur_bar * 4.0 + (cur_pos * 0.25)
                e = s + dur
                global_intervals.append((s, e))

                i += 3
                continue

        i += 1

    attrs: list[list[float]] = []
    for b in range(bars_per_sample):
        note_density = float(onsets[b])
        rhythmic_intensity = float(len(pos_has_onset[b]) / 16.0)
        vel_std = float(np.std(np.asarray(velocities[b], dtype=float), ddof=0)) if velocities[b] else 0.0

        poly_samples: list[int] = []
        for p in range(16):
            t = b * 4.0 + (p * 0.25)
            active = 0
            for s, e in global_intervals:
                if s <= t < e:
                    active += 1
            poly_samples.append(active)
        polyphony_rate = float(sum(poly_samples) / 16.0)

        attrs.append([polyphony_rate, rhythmic_intensity, vel_std, note_density])
    return attrs


def bin_attributes(
    raw_attrs_barsx4: list[list[float]],
    thresholds: dict[str, list[float]],
    bars_per_sample: int = BARS_PER_SAMPLE,
) -> list[list[int]]:
    arr = np.asarray(raw_attrs_barsx4, dtype=float)
    if arr.ndim != 2 or arr.shape != (bars_per_sample, 4):
        raise ValueError(f'Expected shape [{bars_per_sample},4], got {arr.shape}')
    out = np.zeros_like(arr, dtype=int)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        th = np.asarray(thresholds[name], dtype=float)
        if th.shape != (7,):
            raise ValueError(f'Expected 7 thresholds for {name}, got shape {th.shape}')
        out[:, j] = np.digitize(arr[:, j], th, right=False)
    return out.astype(int).tolist()


def load_thresholds(path: str) -> dict[str, list[float]]:
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    th = payload['thresholds']
    for name in ATTRIBUTE_NAMES:
        if name not in th:
            raise ValueError(f'Threshold for {name!r} missing in {path}')
    return th
'''

MMD_CODE = r'''
# ===================== MMD-test additions (not in the repo) =====================
# Fast batched generation with a KV cache, REMI -> MIDI writer, deep-kernel MMD test.

import copy
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- Fast generation ----------

def decoder_of(model_type, model):
    return model if model_type == 'plain' else model.decoder


@torch.no_grad()
def cond_bars_for(model_type, model, attributes, z_p):
    """Per-bar FiLM condition vectors [B, bars, C], exactly as each model's forward builds them."""
    if model_type == 'plain':
        return model.attr(attributes)
    if model_type == 'simple_vae':
        z_g = model.z_proj(z_p)
        z_g_bars = z_g[:, None, :].expand(-1, model.cfg.bars_per_sample, -1)
        return torch.cat([z_g_bars, model.attr(attributes)], dim=-1)
    if model_type == 'vae':
        return torch.cat([model.conductor(z_p), model.attr(attributes)], dim=-1)
    raise ValueError(f'unknown model_type {model_type}')


def build_allowed_table(id_to_token, vocab_index, *, bar_id, pad_id, bos_id, vocab_size, device):
    """allowed[prev, next] = True iff the repo grammar (allowed_next_ids) permits next after prev."""
    table = torch.zeros((vocab_size, vocab_size), dtype=torch.bool)
    for prev in range(vocab_size):
        if prev not in id_to_token:
            continue
        ids = allowed_next_ids(
            prev, id_to_token=id_to_token, vocab_index=vocab_index,
            bar_id=bar_id, pad_id=pad_id, bos_id=bos_id, allow_bar=True,
        )
        table[prev, torch.tensor(ids, dtype=torch.long)] = True
    return table.to(device)


def _block_step(blk, x, cond, k_cache, v_cache, pos):
    """One FiLMBlock forward for tokens at positions [pos, pos+T), reading/writing the KV cache."""
    h = blk.ln1(x)
    gamma, beta = blk.film(cond).chunk(2, dim=-1)
    h = h * (1.0 + gamma) + beta
    attn = blk.attn
    B, T, C = h.shape
    q, k, v = attn.qkv(h).split(C, dim=-1)
    q = q.view(B, T, attn.n_head, attn.head_dim).transpose(1, 2)
    k = k.view(B, T, attn.n_head, attn.head_dim).transpose(1, 2)
    v = v.view(B, T, attn.n_head, attn.head_dim).transpose(1, 2)
    k_cache[:, :, pos:pos + T] = k
    v_cache[:, :, pos:pos + T] = v
    if T > 1 and pos != 0:
        raise ValueError('multi-token steps are only supported from position 0')
    kk = k_cache[:, :, :pos + T].to(q.dtype)
    vv = v_cache[:, :, :pos + T].to(q.dtype)
    y = F.scaled_dot_product_attention(q, kk, vv, attn_mask=None, dropout_p=0.0, is_causal=(T > 1))
    y = y.transpose(1, 2).contiguous().view(B, T, C)
    x = x + attn.proj(y)
    x = x + blk.mlp(blk.ln2(x))
    return x


def decoder_step(dec, tok, pos, cond_tok, caches):
    """tok: [B,T] ids at positions [pos, pos+T); cond_tok: [B,T,C]. Returns logits [B,T,V]."""
    T = tok.shape[1]
    x = dec.tok_emb(tok) + dec.pos_emb.weight[pos:pos + T][None, :, :]
    for blk, (kc, vc) in zip(dec.blocks, caches):
        x = _block_step(blk, x, cond_tok, kc, vc, pos)
    return dec.head(dec.ln_f(x))


def new_caches(dec, B, block_size, device, dtype):
    a = dec.blocks[0].attn
    return [
        (torch.zeros((B, a.n_head, block_size, a.head_dim), device=device, dtype=dtype),
         torch.zeros((B, a.n_head, block_size, a.head_dim), device=device, dtype=dtype))
        for _ in dec.blocks
    ]


def sample_top_p_batched(logits, *, top_p, temperature, generator):
    """Batched version of the repo's sample_next_token: keep the smallest prefix (by prob) whose mass
    reaches top_p, including the first token that crosses it; renormalize; sample."""
    logits = logits.float()
    if float(temperature) != 1.0:
        logits = logits / float(temperature)
    probs = torch.softmax(logits - logits.max(dim=-1, keepdim=True).values, dim=-1)
    sp, si = torch.sort(probs, dim=-1, descending=True)
    cdf = torch.cumsum(sp, dim=-1)
    keep = (cdf - sp) <= float(top_p)
    sp = sp * keep
    sp = sp / sp.sum(dim=-1, keepdim=True)
    choice = torch.multinomial(sp, num_samples=1, generator=generator)
    return si.gather(-1, choice).squeeze(-1)


def bar_index_rule(tokens, *, bar_id, bars_per_sample):
    """Training rule (preprocessing compute_bar_indices): Bar token starts a new bar; tokens before the
    first Bar get bar 0; clamp to bars-1."""
    cur = -1
    out = []
    for t in tokens:
        if int(t) == int(bar_id):
            cur += 1
        out.append(min(max(cur, 0), bars_per_sample - 1))
    return out


@torch.no_grad()
def generate_batch(model_type, model, attributes, *, z_p, allowed_table, bar_id, pad_id,
                   top_p, temperature, generator, block_size=1024, bars_per_sample=8, use_amp=True,
                   amp_dtype=torch.float16, force_amp=False):
    """Generate B excerpts in parallel. Starts from a Bar token at position 0 (as every training chunk
    does), samples with the grammar mask + top-p, stops a row when it samples a 9th Bar or reaches
    block_size tokens. Returns tokens [B, block_size] (PAD after the end), n_bars [B], length [B]."""
    device = attributes.device
    B = attributes.shape[0]
    dec = decoder_of(model_type, model)
    amp = bool(use_amp and (device.type == 'cuda' or force_amp))
    cache_dtype = amp_dtype if amp else torch.float32
    out = torch.full((B, block_size), int(pad_id), dtype=torch.long, device=device)
    out[:, 0] = int(bar_id)
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp):
        cond_bars = cond_bars_for(model_type, model, attributes, z_p)
        C = cond_bars.shape[-1]
        caches = new_caches(dec, B, block_size, device, cache_dtype)
        last = out[:, 0].clone()
        cur_bar = torch.zeros(B, dtype=torch.long, device=device)
        n_bars = torch.ones(B, dtype=torch.long, device=device)
        length = torch.full((B,), block_size, dtype=torch.long, device=device)
        active = torch.ones(B, dtype=torch.bool, device=device)
        for pos in range(block_size - 1):
            cond = cond_bars.gather(1, cur_bar.view(B, 1, 1).expand(B, 1, C))
            logits = decoder_step(dec, last.view(B, 1), pos, cond, caches)[:, -1].float()
            logits = logits.masked_fill(~allowed_table[last], float('-inf'))
            nxt = sample_top_p_batched(logits, top_p=top_p, temperature=temperature, generator=generator)
            is_bar = nxt == int(bar_id)
            stop = active & is_bar & (n_bars >= bars_per_sample)
            length = torch.where(stop, torch.full_like(length, pos + 1), length)
            active = active & ~stop
            write = active
            out[:, pos + 1] = torch.where(write, nxt, out[:, pos + 1])
            new_bar = write & is_bar
            n_bars = n_bars + new_bar.long()
            cur_bar = torch.where(new_bar, (cur_bar + 1).clamp(max=bars_per_sample - 1), cur_bar)
            last = torch.where(write, nxt, last)
            if (pos & 15) == 15 and not bool(active.any()):
                break
    return out, n_bars, length


@torch.no_grad()
def kv_equivalence_check(model_type, model, X, bar_indices, attributes, *, z_p, n_tokens, bar_id):
    """Compare KV-cached step-by-step logits with the model's own full forward (fp32). Returns max |diff|
    and whether our bar-index rule reproduces the memmap's bar_indices."""
    device = X.device
    B = X.shape[0]
    if model_type == 'plain':
        full = model(X, bar_indices=bar_indices, attributes=attributes)['logits']
    else:
        full = model(X, bar_indices=bar_indices, attributes=attributes, z_p=z_p)['logits']
    dec = decoder_of(model_type, model)
    cond_bars = cond_bars_for(model_type, model, attributes, z_p)
    C = cond_bars.shape[-1]
    caches = new_caches(dec, B, X.shape[1], device, torch.float32)
    diffs = []
    for pos in range(n_tokens):
        bi = bar_indices[:, pos].long()
        cond = cond_bars.gather(1, bi.view(B, 1, 1).expand(B, 1, C))
        lg = decoder_step(dec, X[:, pos:pos + 1], pos, cond, caches)[:, -1]
        diffs.append(float((lg - full[:, pos]).abs().max()))
    rule_ok = True
    Xc = X.cpu().numpy()
    bic = bar_indices.cpu().numpy()
    for b in range(B):
        L = int((Xc[b] != 0).sum())
        rule = bar_index_rule(Xc[b, :L].tolist(), bar_id=bar_id, bars_per_sample=8)
        rule_ok = rule_ok and (rule == bic[b, :L].tolist())
    return max(diffs), rule_ok


# ---------- REMI tokens -> MIDI ----------

def remi_tokens_to_midi(tokens, id_to_token, path, *, ticks_per_beat=480, positions_per_beat=4):
    """Minimal REMI decoder (4/4, 16 positions per bar). The same function writes real and generated
    excerpts, so file format cannot differ between the two sides of a test."""
    import mido
    toks = [id_to_token.get(int(t), '') for t in tokens]
    tick_per_pos = ticks_per_beat // positions_per_beat
    bar_ticks = 4 * ticks_per_beat
    events = []  # (tick, priority, msg); priority: tempo 0, note_off 1, note_on 2
    cur_bar = -1
    pos = 0
    i = 0
    n = len(toks)
    n_notes = 0
    while i < n:
        s = toks[i]
        head = s.split('_', 1)[0]
        if head == 'PAD':
            break
        if head == 'Bar':
            cur_bar += 1
            pos = 0
        elif head == 'Position':
            pos = int(s.split('_', 1)[1])
        elif cur_bar >= 0 and head == 'Tempo':
            t0 = cur_bar * bar_ticks + pos * tick_per_pos
            bpm = float(s.split('_', 1)[1])
            events.append((t0, 0, mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(bpm), time=0)))
        elif cur_bar >= 0 and head == 'Pitch' and i + 2 < n \
                and toks[i + 1].startswith('Velocity_') and toks[i + 2].startswith('Duration_'):
            pitch = int(s.split('_', 1)[1])
            vel = max(1, min(127, int(toks[i + 1].split('_', 1)[1])))
            dur = max(1, int(round(_parse_duration_beats(toks[i + 2]) * ticks_per_beat)))
            t0 = cur_bar * bar_ticks + pos * tick_per_pos
            events.append((t0, 2, mido.Message('note_on', note=pitch, velocity=vel, time=0)))
            events.append((t0 + dur, 1, mido.Message('note_off', note=pitch, velocity=0, time=0)))
            n_notes += 1
            i += 3
            continue
        i += 1
    events.sort(key=lambda e: (e[0], e[1]))
    mid = mido.MidiFile(ticks_per_beat=ticks_per_beat)
    track = mido.MidiTrack()
    mid.tracks.append(track)
    track.append(mido.Message('program_change', program=0, time=0))
    prev = 0
    for tick, _p, msg in events:
        track.append(msg.copy(time=int(tick - prev)))
        prev = tick
    track.append(mido.MetaMessage('end_of_track', time=0))
    mid.save(str(path))
    return n_notes


# ---------- Deep-kernel MMD (Liu et al. 2020, MMD-D; follows the official DK-for-TST code) ----------

class PhiNet(nn.Module):
    """ModelLatentF from DK-for-TST: 4 linear layers, softplus between."""

    def __init__(self, x_in, H, x_out):
        super().__init__()
        self.latent = nn.Sequential(
            nn.Linear(x_in, H), nn.Softplus(),
            nn.Linear(H, H), nn.Softplus(),
            nn.Linear(H, H), nn.Softplus(),
            nn.Linear(H, x_out),
        )

    def forward(self, x):
        return self.latent(x)


def pdist2(x, y):
    xx = (x * x).sum(1, keepdim=True)
    yy = (y * y).sum(1, keepdim=True)
    return (xx + yy.t() - 2.0 * x @ y.t()).clamp_min(0.0)


class DeepKernel(nn.Module):
    """k(x,y) = [(1-eps) exp(-|phi x - phi y|^2 / sigma0) + eps] exp(-|x-y|^2 / sigma)
    = (1-eps) exp(-Dphi/sigma0 - D/sigma) + eps exp(-D/sigma)   (official code form).
    eps = sigmoid(eps_raw) in (0,1); sigma = sigma_raw^2; sigma0 = sigma0_raw^2."""

    def __init__(self, d_in, width, *, sigma_init, sigma0_init):
        super().__init__()
        self.phi = PhiNet(d_in, width, width)
        self.eps_raw = nn.Parameter(torch.tensor(math.log(1e-10)))
        self.sigma_raw = nn.Parameter(torch.tensor(math.sqrt(sigma_init)))
        self.sigma0_raw = nn.Parameter(torch.tensor(math.sqrt(sigma0_init)))

    def params(self):
        return torch.sigmoid(self.eps_raw), self.sigma_raw ** 2, self.sigma0_raw ** 2

    def gram(self, S):
        """Full kernel matrix on pooled inputs S [m, d]."""
        ep, sigma, sigma0 = self.params()
        Fs = self.phi(S)
        D_org = pdist2(S, S)
        D_phi = pdist2(Fs, Fs)
        return (1 - ep) * torch.exp(-D_phi / sigma0 - D_org / sigma) + ep * torch.exp(-D_org / sigma)


def cross_gram(kernel, Sa, Sb):
    """Kernel block k(Sa_i, Sb_j) without forming the pooled gram (saves memory at large n)."""
    ep, sigma, sigma0 = kernel.params()
    D_org = pdist2(Sa, Sb)
    D_phi = pdist2(kernel.phi(Sa), kernel.phi(Sb))
    return (1 - ep) * torch.exp(-D_phi / sigma0 - D_org / sigma) + ep * torch.exp(-D_org / sigma)


def _offdiag_mean(K):
    n = K.shape[0]
    return (K.sum() - K.diagonal().sum()) / (n * (n - 1))


def _ustat_and_var(H):
    """U-statistic mean of H over i != j and the eq.-5 variance estimate (Var(estimate) ~ var / n)."""
    n = H.shape[0]
    est = _offdiag_mean(H)
    r = H.sum(1) / n
    var = 4 * (torch.dot(r, r) / n - (H.sum() / n / n) ** 2)
    return est, var


def _clustered_var_of_sum(e, groups):
    """Var(sum_i e_i) allowing correlation within each grouping (Cameron-Gelbach-Miller multiway form).
    groups: list of LongTensors of group ids (one per clustering dimension)."""
    def ss(ids):
        _, inv = torch.unique(ids, return_inverse=True)
        sums = torch.zeros(int(inv.max()) + 1, dtype=e.dtype, device=e.device).index_add_(0, inv, e)
        return float((sums * sums).sum())
    if len(groups) == 1:
        return ss(groups[0])
    g0, g1 = groups
    inter = g0 * (int(g1.max()) + 1) + g1
    v = ss(g0) + ss(g1) - ss(inter)
    return max(v, ss(g0), ss(g1))  # CGM can go negative in small samples; never go below one-way


@torch.no_grad()
def relative_similarity(kernel, Q, B, pairs, *, block_sets=None):
    """Relative similarity test (after Bounliphone et al. 2016) with one kernel shared by all sets.
    Q: {label: [n, d]} generated sets, index-aligned (row i of every set was conditioned on the same grid);
    B: [n, d] real reference. Row i of each set is paired with B row i, and
    D = MMD^2_u(Q1, B) - MMD^2_u(Q2, B) is a U-statistic with kernel H1 - H2 (the k(b_i, b_j) terms cancel).
    Linearization: D - E[D] ~ (2/n) sum_i e_i, e_i = (row mean of H1 - H2) - D.
      z_naive:   rows independent (eq.-5 style variance).
      z_cluster_<w>: rows correlated within piece blocks of the conditioning chunk (shared by all Q sets) and
                 within piece blocks of the real chunk; two-way cluster-robust variance, one per block set
                 in block_sets = {name: (blocks_q, blocks_b)}.
      z_cluster: the most conservative (smallest |z|) over the block sets and the naive SE; decisions use this.
    Two-sided normal p-values throughout."""
    n = B.shape[0]
    Kbb = cross_gram(kernel, B, B)
    H = {}
    absolute = {}
    for lab, X in Q.items():
        assert X.shape[0] == n
        Kxx = cross_gram(kernel, X, X)
        Kxb = cross_gram(kernel, X, B)
        H[lab] = Kxx + Kbb - Kxb - Kxb.t()
        m2, v = _ustat_and_var(H[lab])
        absolute[lab] = {'mmd2': float(m2), 'J': float(m2 / torch.sqrt(v + 1e-8))}
        del Kxx, Kxb
    group_sets = {}
    for name, (bq, bb) in (block_sets or {}).items():
        group_sets[name] = [torch.as_tensor(bq, dtype=torch.long, device=B.device),
                            torch.as_tensor(bb, dtype=torch.long, device=B.device)]
    out = {}
    for a, b in pairs:
        HD = H[a] - H[b]
        D = float(_offdiag_mean(HD))
        e = (HD.sum(1) - HD.diagonal()) / (n - 1) - D
        var_naive = 4.0 / n ** 2 * float((e * e).sum())
        se_naive = math.sqrt(max(var_naive, 1e-30))
        z_naive = D / se_naive
        res = {'D': D, 'se_naive': se_naive, 'z_naive': z_naive,
               'p_naive': math.erfc(abs(z_naive) / math.sqrt(2.0))}
        worst = None
        for name, groups in group_sets.items():
            var_cl = 4.0 / n ** 2 * _clustered_var_of_sum(e.double(), groups)
            se_cl = math.sqrt(max(var_cl, 1e-30))
            z_cl = D / se_cl
            res[f'z_cluster_{name}'] = z_cl
            res[f'design_effect_{name}'] = var_cl / max(var_naive, 1e-30)
            if worst is None or se_cl > worst:
                worst = se_cl
        if worst is not None:
            worst = max(worst, se_naive)  # never less conservative than independent rows
            z_cl = D / worst
            res.update(se_cluster=worst, z_cluster=z_cl, p_cluster=math.erfc(abs(z_cl) / math.sqrt(2.0)),
                       design_effect=worst ** 2 / max(var_naive, 1e-30))
        out[f'{a}-{b}'] = res
    return {'absolute': absolute, 'pairs': out}


def fixed_gaussian_kernel(S_fit, d_in, width):
    """Plain Gaussian kernel exp(-|x-y|^2 / sigma) on the (standardized) embeddings, sigma = median squared
    distance on S_fit. Built as a DeepKernel with eps = 1 so the learned phi term is switched off."""
    k = DeepKernel(d_in, width, sigma_init=1.0, sigma0_init=1.0).to(S_fit.device)
    with torch.no_grad():
        D = pdist2(S_fit, S_fit)
        med = float(D[D > 0].median())
        k.sigma_raw.fill_(math.sqrt(med))
        k.eps_raw.fill_(50.0)  # sigmoid(50) == 1.0 in float32, so (1 - eps) == 0
    assert float((1 - k.params()[0]).detach()) == 0.0
    return k


def mmd2_u_and_var(K, n):
    """Equal-size U-statistic MMD^2 (eq. 2) and the V-statistic variance estimate (eq. 5), from a pooled
    gram K of [X; Y] with |X| = |Y| = n. Mirrors h1_mean_var_gram(use_1sample_U=True)."""
    Kx = K[:n, :n]
    Ky = K[n:, n:]
    Kxy = K[:n, n:]
    xx = (Kx.sum() - Kx.diagonal().sum()) / (n * (n - 1))
    yy = (Ky.sum() - Ky.diagonal().sum()) / (n * (n - 1))
    xy = (Kxy.sum() - Kxy.diagonal().sum()) / (n * (n - 1))
    mmd2 = xx - 2 * xy + yy
    hh = Kx + Ky - Kxy - Kxy.t()
    r = hh.sum(1) / n
    V1 = torch.dot(r, r) / n
    V2 = hh.sum() / n / n
    var = 4 * (V1 - V2 ** 2)
    return mmd2, var


def j_stat(mmd2, var, lam=1e-8):
    """J_lambda = MMD_u^2 / sqrt(var + lambda)   (eq. 4: divide by the std, not the variance)."""
    return mmd2 / torch.sqrt(var + lam)


def _mmd2_from_index(K, ix, iy):
    n = ix.numel()
    Kx = K[ix][:, ix]
    Ky = K[iy][:, iy]
    Kxy = K[ix][:, iy]
    xx = (Kx.sum() - Kx.diagonal().sum()) / (n * (n - 1))
    yy = (Ky.sum() - Ky.diagonal().sum()) / (n * (n - 1))
    xy = (Kxy.sum() - Kxy.diagonal().sum()) / (n * (n - 1))
    return xx - 2 * xy + yy


@torch.no_grad()
def permutation_test(kernel, X, Y, *, n_perm, generator):
    """Phase 2 of Algorithm 1 with a frozen kernel. p = (1 + #{perm >= est}) / (1 + n_perm)."""
    n = X.shape[0]
    S = torch.cat([X, Y], 0)
    K = kernel.gram(S)
    est, var = mmd2_u_and_var(K, n)
    J = j_stat(est, var)
    count = 0
    for _ in range(int(n_perm)):
        perm = torch.randperm(2 * n, generator=generator, device=S.device)
        if float(_mmd2_from_index(K, perm[:n], perm[n:])) >= float(est):
            count += 1
    p = (1 + count) / (1 + int(n_perm))
    return {'mmd2': float(est), 'var': float(var), 'J': float(J), 'p': float(p)}


class Standardizer:
    def __init__(self, Xfit):
        self.mean = Xfit.mean(0, keepdim=True)
        self.std = Xfit.std(0, keepdim=True).clamp_min(1e-6)

    def __call__(self, X):
        return (X - self.mean) / self.std


def train_deep_kernel(Xfit, Yfit, Xval, Yval, *, width, lr, steps, val_every, seed, early_stop=True, patience=None):
    """Phase 1 of Algorithm 1: maximize J on the fit sets (full batch, Adam). With early_stop, keep the
    parameters with the best validation J (validation sets never include test points). With patience,
    training stops once validation J has not improved for that many steps."""
    torch.manual_seed(int(seed))
    n = Xfit.shape[0]
    d = Xfit.shape[1]
    S_fit = torch.cat([Xfit, Yfit], 0)
    S_val = torch.cat([Xval, Yval], 0)
    nv = Xval.shape[0]
    kernel = DeepKernel(d, width, sigma_init=2.0 * d, sigma0_init=1.0).to(Xfit.device)
    with torch.no_grad():  # median heuristic for the phi-space lengthscale at initialization
        Fs = kernel.phi(S_fit)
        D = pdist2(Fs, Fs)
        med = float(D[D > 0].median()) if bool((D > 0).any()) else 1.0
        kernel.sigma0_raw.fill_(math.sqrt(max(med, 1e-6)))
    opt = torch.optim.Adam(kernel.parameters(), lr=float(lr))
    log = {'fit_J': [], 'val_J': [], 'val_step': []}
    best_val = -float('inf')
    best_state = copy.deepcopy(kernel.state_dict())
    best_step = 0
    for t in range(int(steps)):
        mmd2, var = mmd2_u_and_var(kernel.gram(S_fit), n)
        J = j_stat(mmd2, var)
        opt.zero_grad()
        (-J).backward()
        opt.step()
        log['fit_J'].append(float(J.detach()))
        if (t + 1) % int(val_every) == 0 or t + 1 == int(steps):
            with torch.no_grad():
                vm, vv = mmd2_u_and_var(kernel.gram(S_val), nv)
                vJ = float(j_stat(vm, vv))
            log['val_J'].append(vJ)
            log['val_step'].append(t + 1)
            if vJ > best_val:
                best_val = vJ
                best_state = copy.deepcopy(kernel.state_dict())
                best_step = t + 1
            elif early_stop and patience and (t + 1) - best_step >= int(patience):
                break
    if early_stop:
        kernel.load_state_dict(best_state)
    else:
        best_step = int(steps)
    with torch.no_grad():
        fm, fv = mmd2_u_and_var(kernel.gram(S_fit), n)
        fit_J_final = float(j_stat(fm, fv))
        vm, vv = mmd2_u_and_var(kernel.gram(S_val), nv)
        val_J_final = float(j_stat(vm, vv))
    ep, sigma, sigma0 = kernel.params()
    info = {
        'best_step': best_step, 'steps_run': len(log['fit_J']), 'fit_J': fit_J_final, 'val_J': val_J_final,
        'eps': float(ep.detach()), 'sigma': float(sigma.detach()), 'sigma0': float(sigma0.detach()),
        'fit_J_curve_every_val': log['fit_J'][int(val_every) - 1::int(val_every)],
        'val_J_curve': log['val_J'],
    }
    return kernel, info
'''

MODULE_CODE = '\n\n'.join([
    'from __future__ import annotations',
    PLAIN_MODEL_CODE,
    SIMPLE_MODEL_CODE,
    VAE_MODEL_CODE,
    GRAMMAR_CODE,
    ATTR_CODE,
    MMD_CODE,
])

MODULE_PATH.write_text(MODULE_CODE, encoding='utf-8')
print('Wrote module:', MODULE_PATH)

if str(Path('.').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('.').resolve()))

if 'musicgen_kaggle_mmd_min' in sys.modules:
    mg = importlib.reload(sys.modules['musicgen_kaggle_mmd_min'])
else:
    import musicgen_kaggle_mmd_min as mg
print('Imported self-contained module:', mg.__file__)

In [ ]:
# -------------------------
# Tokenizer metadata, attribute thresholds, grammar table
# -------------------------

tok_meta = json.loads(TOKENIZER_META_PATH.read_text(encoding='utf-8'))
if isinstance(tok_meta['id_to_token'], dict):
    id_to_token = {int(k): v for k, v in tok_meta['id_to_token'].items()}
else:
    id_to_token = {int(i): s for i, s in enumerate(tok_meta['id_to_token'])}

pad_id = int(tok_meta['pad_id'])
bos_id = int(tok_meta['bos_id'])
bar_id = int(tok_meta['bar_id'])
VOCAB_SIZE = max(id_to_token.keys()) + 1

thresholds = json.loads(QUANTILES_PATH.read_text(encoding='utf-8'))['thresholds']
vocab_index = mg.build_vocab_index(id_to_token)
ALLOWED = mg.build_allowed_table(
    id_to_token, vocab_index, bar_id=bar_id, pad_id=pad_id, bos_id=bos_id,
    vocab_size=VOCAB_SIZE, device=DEVICE,
)

# MIDI writer assumptions: 4/4, 16 positions per bar, parseable durations.
n_pos = sum(1 for s in id_to_token.values() if s.startswith('Position_'))
assert n_pos == 16, f'expected 16 Position tokens, got {n_pos}'
for s in id_to_token.values():
    if s.startswith('Duration_'):
        mg._parse_duration_beats(s)

print('pad_id:', pad_id, 'bos_id:', bos_id, 'bar_id:', bar_id, '| vocab:', VOCAB_SIZE)
print('grammar table:', tuple(ALLOWED.shape), '| allowed after Bar:', int(ALLOWED[bar_id].sum()))

In [ ]:
# -------------------------
# Load checkpoints + instantiate models (as notebook 08)
# -------------------------

def _load_ckpt(path: Path) -> dict:
    ckpt = torch.load(path, map_location='cpu')
    if isinstance(ckpt, dict) and 'cfg' in ckpt:
        return ckpt
    raise ValueError(f'Unexpected ckpt format at {path}')


def _state_dict(ckpt: dict):
    sd = ckpt.get('model') or ckpt.get('model_state_dict') or ckpt.get('state_dict')
    if sd is None:
        raise KeyError('ckpt missing model state dict')
    return sd


def _instantiate(model_type: str, ckpt: dict):
    cfg_cls, model_cls = {
        'plain': (mg.PlainTransformerConfig, mg.PlainTransformerDecoder),
        'simple_vae': (mg.SimpleVAEConfig, mg.SimpleMusicVAE),
        'vae': (mg.VAEConfig, mg.MusicVAE),
    }[model_type]
    cfg = cfg_cls(**ckpt['cfg']) if isinstance(ckpt['cfg'], dict) else ckpt['cfg']
    m = model_cls(cfg)
    m.load_state_dict(_state_dict(ckpt), strict=True)
    return m


MODELS = {}
for mt in MODEL_TYPES:
    MODELS[mt] = _instantiate(mt, _load_ckpt(CKPT[mt])).to(DEVICE).eval()
    n_params = sum(p.numel() for p in MODELS[mt].parameters())
    print(f'loaded {LABEL[mt]:>6} ({n_params / 1e6:.1f}M params)')

Z_DIM = {mt: int(getattr(MODELS[mt].cfg, 'z_dim', 0)) for mt in MODEL_TYPES}
print('z_dim:', Z_DIM)

In [ ]:
# -------------------------
# Open val/test memmaps; check KV-cached decoding against each model's own forward
# -------------------------

def open_split(split_dir: Path):
    meta = json.loads((split_dir / 'meta.json').read_text(encoding='utf-8'))
    n = int(meta['num_examples'])
    t = int(meta.get('tokens_per_example', 1024))
    b = int(meta.get('bars_per_sample', 8))
    X = np.memmap(split_dir / 'X.u16.memmap', mode='r', dtype=np.uint16, shape=(n, t))
    BI = np.memmap(split_dir / 'bar_indices.u8.memmap', mode='r', dtype=np.uint8, shape=(n, t))
    A = np.memmap(split_dir / 'attributes.u8.memmap', mode='r', dtype=np.uint8, shape=(n, b, 4))
    return meta, X, BI, A


val_meta, VAL_X, VAL_BI, VAL_A = open_split(VAL_DIR)
test_meta, TEST_X, TEST_BI, TEST_A = open_split(TEST_DIR)
print('val :', VAL_X.shape, '| pieces:', val_meta.get('pieces_seen'))
print('test:', TEST_X.shape, '| pieces:', test_meta.get('pieces_seen'))

idx = [0, 1000, 5000, 10000]
Xc = torch.from_numpy(np.asarray(TEST_X[idx], dtype=np.int64)).to(DEVICE)
BIc = torch.from_numpy(np.asarray(TEST_BI[idx], dtype=np.int64)).to(DEVICE)
Ac = torch.from_numpy(np.asarray(TEST_A[idx], dtype=np.int64)).to(DEVICE)
assert bool((Xc[:, 0] == bar_id).all()), 'real chunks are expected to start with Bar'

KV_CHECK = {}
gchk = torch.Generator(device=DEVICE).manual_seed(123)
for mt in MODEL_TYPES:
    z = torch.randn((len(idx), Z_DIM[mt]), generator=gchk, device=DEVICE) if Z_DIM[mt] else None
    diff, rule_ok = mg.kv_equivalence_check(
        mt, MODELS[mt], Xc, BIc, Ac, z_p=z, n_tokens=int(CFG['KV_CHECK_TOKENS']), bar_id=bar_id,
    )
    KV_CHECK[LABEL[mt]] = {'max_abs_logit_diff': diff, 'bar_rule_matches_memmap': bool(rule_ok)}
    print(f'{LABEL[mt]:>6}: max |logit diff| = {diff:.2e} | bar-index rule matches memmap: {rule_ok}')
    assert rule_ok, 'bar-index rule does not reproduce the memmap bar_indices'
    assert diff < float(CFG['KV_CHECK_TOL']), f'KV-cached decoding differs from full forward for {mt}'
print('KV-cache check passed: cached decoding reproduces each model forward.')

In [ ]:
# -------------------------
# Draw real sets
#   A: from val (conditions for generation; control side)
#   B: from test (real reference)
# Evenly spaced grid over each memmap (spreads the draw across pieces), then a fixed random order,
# so the first N rows are a random subset of the grid for any N the pilot chooses.
# -------------------------

def draw_grid(n_total: int, k: int, rng) -> np.ndarray:
    grid = np.unique(np.linspace(0, n_total - 1, k).round().astype(np.int64))
    return rng.permutation(grid)


rng_draw = np.random.default_rng(SEED)
A_IDX = draw_grid(VAL_X.shape[0], int(CFG['N_MAX']), rng_draw)
B_IDX = draw_grid(TEST_X.shape[0], int(CFG['N_MAX']), rng_draw)

A_TOK = np.asarray(VAL_X[np.sort(A_IDX)], dtype=np.uint16)[np.argsort(np.argsort(A_IDX))]
A_ATTR = np.asarray(VAL_A[np.sort(A_IDX)], dtype=np.int64)[np.argsort(np.argsort(A_IDX))]
B_TOK = np.asarray(TEST_X[np.sort(B_IDX)], dtype=np.uint16)[np.argsort(np.argsort(B_IDX))]
assert np.array_equal(A_TOK[0], VAL_X[A_IDX[0]]) and np.array_equal(B_TOK[-1], TEST_X[B_IDX[-1]])

def tok_len(T):
    return (T != pad_id).sum(axis=1)

print('A:', A_TOK.shape, '| mean len', tok_len(A_TOK).mean().round(1))
print('B:', B_TOK.shape, '| mean len', tok_len(B_TOK).mean().round(1))

In [ ]:
# -------------------------
# CLaMP 2 extractor (downloads weights on first use) + MIDI round-trip check + embedding pilot
# -------------------------

t_load = time.time()
EXTRACTOR = CLaMP2Extractor(verbose=True)
print(f'CLaMP 2 loaded in {fmt_secs(time.time() - t_load)}')

MIDI_TMP = WORKING_ROOT / 'midi_tmp'
MIDI_TMP.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR = OUT_DIR / 'midi_samples'
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)


def embed_tokens(tokens_row, name: str) -> np.ndarray:
    path = MIDI_TMP / f'{name}.mid'
    mg.remi_tokens_to_midi(tokens_row, id_to_token, path)
    feat = EXTRACTOR.extract_feature(path)
    return np.asarray(feat, dtype=np.float32).reshape(-1), path


# Round trip: written MIDI parses with mido and keeps every note.
p_rt = MIDI_TMP / 'roundtrip.mid'
n_written = mg.remi_tokens_to_midi(A_TOK[0], id_to_token, p_rt)
n_read = sum(1 for msg in mido.MidiFile(str(p_rt)) if msg.type == 'note_on' and msg.velocity > 0)
n_tok = int(sum(1 for t in A_TOK[0] if id_to_token.get(int(t), '').startswith('Pitch_')))
print(f'MIDI round trip: pitch tokens {n_tok}, written {n_written}, read back {n_read}')
assert n_written == n_tok == n_read

for i in range(3):  # warm-up, untimed (first calls are much slower)
    _f, _p = embed_tokens(B_TOK[i], f'warmup_{i}')
    _p.unlink()
t_e = time.time()
k = int(CFG['EMB_PILOT_FILES'])
pilot_feats = []
for i in range(k):
    f, pth = embed_tokens(B_TOK[i], f'pilot_{i}')
    pilot_feats.append(f)
    pth.unlink()
SEC_PER_FILE = (time.time() - t_e) / k
EMB_DIM = pilot_feats[0].shape[0]
print(f'embedding dim {EMB_DIM} | {SEC_PER_FILE:.3f} s per file (MIDI write + CLaMP 2)')

In [ ]:
# -------------------------
# Generation helpers + timing pilot (first batch of every model) -> choose N
# -------------------------

GEN = {lab: {'tokens': [], 'n_bars': [], 'length': [], 'secs': 0.0, 'done': 0} for lab in GEN_SPECS}
SEED_OFFSET = {'plain': 0, 'simple': 1, 'hier': 2, 'plain2': 3}


def _sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def run_batch(lab: str, b: int, lo: int, hi: int) -> float:
    mt = GEN_SPECS[lab]
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(SEED * 1_000_003 + SEED_OFFSET[lab] * 10_007 + b)
    attrs = torch.from_numpy(A_ATTR[lo:hi]).to(DEVICE)
    z = torch.randn((hi - lo, Z_DIM[mt]), generator=gen, device=DEVICE) if Z_DIM[mt] else None
    _sync()
    t = time.time()
    toks, n_bars, length = mg.generate_batch(
        mt, MODELS[mt], attrs, z_p=z, allowed_table=ALLOWED, bar_id=bar_id, pad_id=pad_id,
        top_p=TOP_P, temperature=TEMPERATURE, generator=gen, block_size=BLOCK_SIZE, bars_per_sample=BARS,
    )
    _sync()
    dt = time.time() - t
    G = GEN[lab]
    G['tokens'].append(toks.to('cpu', torch.int32).numpy().astype(np.uint16))
    G['n_bars'].append(n_bars.cpu().numpy())
    G['length'].append(length.cpu().numpy())
    G['secs'] += dt
    G['done'] = hi
    return dt


bs = int(CFG['GEN_BATCH'])
pilot_n = min(bs, int(CFG['N_MAX']))
SEC_PER_EXC = {}
for lab in GEN_SPECS:
    dt = run_batch(lab, 0, 0, pilot_n)
    SEC_PER_EXC[lab] = dt / pilot_n
    nb = GEN[lab]['n_bars'][0]
    print(f'pilot {lab:>6}: {dt:6.1f}s for {pilot_n} | {SEC_PER_EXC[lab]:.3f} s/excerpt | '
          f'8-bar rate {(nb == BARS).mean():.2f} | mean len {GEN[lab]["length"][0].mean():.0f}')

gen_rate = sum(SEC_PER_EXC.values())          # seconds per index, all generated sets
emb_rate = (2 + len(GEN_SPECS)) * SEC_PER_FILE  # A, B and every generated set
remaining = CFG['DEADLINE_S'] - elapsed() - CFG['TEST_RESERVE_S']
n_by_gen = CFG['GEN_BUDGET_S'] / gen_rate
n_by_emb = CFG['EMB_BUDGET_S'] / emb_rate
n_by_total = remaining / (gen_rate + emb_rate)
N = int(max(pilot_n, min(CFG['N_MAX'], n_by_gen, n_by_emb, n_by_total)))
if N < CFG['N_FLOOR']:
    print(f'WARNING: time budget limits N to {N} (< floor {CFG["N_FLOOR"]}).')

proj_gen = N * gen_rate
proj_emb = N * emb_rate
print(f'limits: gen {n_by_gen:.0f}, emb {n_by_emb:.0f}, total {n_by_total:.0f} -> N = {N} per model')
print(f'projected: generation {fmt_secs(proj_gen)}, embedding {fmt_secs(proj_emb)}, '
      f'tests <= {fmt_secs(CFG["TEST_RESERVE_S"])}; finish about {fmt_secs(elapsed() + proj_gen + proj_emb + CFG["TEST_RESERVE_S"])} '
      f'after start (deadline {fmt_secs(CFG["DEADLINE_S"])})')
TIMINGS = {'pilot_sec_per_excerpt': dict(SEC_PER_EXC), 'sec_per_file': SEC_PER_FILE}

In [ ]:
# -------------------------
# Generation: N excerpts per model, batches round-robin over models (a hard stop leaves equal counts)
# -------------------------

t_gen = time.time()
gen_stop = CFG['GEN_BUDGET_S'] * 1.25
batches = [(b, b * bs, min(N, (b + 1) * bs)) for b in range(1, math.ceil(N / bs))]
n_sets = len(GEN_SPECS)
pbar = tqdm(total=n_sets * N, initial=n_sets * min(N, pilot_n), desc=f'generate ({n_sets} sets)', unit='exc')
hard_stopped = False
for b, lo, hi in batches:
    # stop if over the generation budget, or if embedding everything generated so far would eat into the test reserve
    if time.time() - t_gen > gen_stop or elapsed() + (hi - lo) * gen_rate + hi * emb_rate > CFG['DEADLINE_S'] - CFG['TEST_RESERVE_S']:
        hard_stopped = True
        print(f'hard stop at index {lo}: generation budget reached')
        break
    for lab in GEN_SPECS:
        run_batch(lab, b, lo, hi)
        pbar.update(hi - lo)
    pbar.set_postfix({lab: f'{GEN[lab]["secs"] / GEN[lab]["done"]:.2f}s/exc' for lab in GEN_SPECS})
pbar.close()

N_DONE = min(GEN[lab]['done'] for lab in GEN_SPECS)
Q_TOK, Q_BARS, Q_LEN = {}, {}, {}
for lab in GEN_SPECS:
    Q_TOK[lab] = np.concatenate(GEN[lab]['tokens'])[:N_DONE]
    Q_BARS[lab] = np.concatenate(GEN[lab]['n_bars'])[:N_DONE]
    Q_LEN[lab] = np.concatenate(GEN[lab]['length'])[:N_DONE]
TIMINGS['generation_s'] = sum(GEN[lab]['secs'] for lab in GEN_SPECS)
TIMINGS['gen_sec_per_excerpt'] = {lab: GEN[lab]['secs'] / GEN[lab]['done'] for lab in GEN_SPECS}
print(f'generated {N_DONE} per set in {fmt_secs(TIMINGS["generation_s"])} | hard stop: {hard_stopped}')

In [ ]:
# -------------------------
# Valid excerpts, common size, attribute shift, lengths
#   valid = exactly 8 bars. A and every Q_m stay index-aligned (excerpt i was conditioned on A_i),
#   so the kept index set is the intersection over models; B is subsampled to the same size.
# -------------------------

valid = {lab: Q_BARS[lab] == BARS for lab in GEN_SPECS}
DROPPED = {lab: int((~valid[lab]).sum()) for lab in GEN_SPECS}
keep = np.where(np.logical_and.reduce([valid[lab] for lab in GEN_SPECS]))[0]
N_ALL = int(len(keep))
rng_sub = np.random.default_rng(SEED + 1)
keep_B = np.sort(rng_sub.choice(N_DONE, size=N_ALL, replace=False))

SETS = {'A': A_TOK[:N_DONE][keep], 'B': B_TOK[:N_DONE][keep_B]}
for lab in GEN_SPECS:
    SETS[lab] = Q_TOK[lab][keep]
ATTR_REQ = A_ATTR[:N_DONE][keep]
# Piece-proxy blocks for cluster-robust SEs: chunks are stored piece by piece, so consecutive memmap indices
# share a piece (pitch-class similarity decays to the random-pair level by about 100 indices).
MM_A = A_IDX[:N_DONE][keep].astype(np.int64)     # memmap index (val) of each kept A / Q row
MM_B = B_IDX[:N_DONE][keep_B].astype(np.int64)   # memmap index (test) of each kept B row
for w in CFG['BLOCK_WIDTHS']:
    print(f'SE blocks of {w} memmap indices: A {len(np.unique(MM_A // w))}, B {len(np.unique(MM_B // w))}')
w = int(CFG['SPLIT_BLOCK_WIDTH'])
print(f'split blocks of {w}: A {len(np.unique(MM_A // w))}, B {len(np.unique(MM_B // w))}')
print('dropped (fewer than 8 bars):', DROPPED, '| common size N_ALL =', N_ALL)


def binned_attrs(tokens_2d) -> np.ndarray:
    out = np.zeros((len(tokens_2d), BARS, 4), dtype=np.int64)
    for i, row in enumerate(tqdm(tokens_2d, desc='recompute attrs', leave=False)):
        raw = mg.compute_raw_attributes_per_bar_from_tokens(
            [int(t) for t in row], id_to_token=id_to_token, bar_token_id=bar_id, bars_per_sample=BARS,
        )
        out[i] = np.asarray(mg.bin_attributes(raw, thresholds, bars_per_sample=BARS))
    return out


ATTR_NAMES = ['polyphony_rate', 'rhythmic_intensity', 'velocity_dynamics', 'note_density']
ATTR_DIAG = {}
real_check = binned_attrs(SETS['A'][: min(200, N_ALL)])
ATTR_DIAG['A_recompute_exact_match'] = float((real_check == ATTR_REQ[: len(real_check)]).mean())
print(f'pipeline check: recomputed vs stored bins on real A: {ATTR_DIAG["A_recompute_exact_match"]:.3f} exact')
for lab in GEN_SPECS:
    got = binned_attrs(SETS[lab])
    d = got - ATTR_REQ
    ATTR_DIAG[lab] = {
        'mean_shift': {n: float(d[..., j].mean()) for j, n in enumerate(ATTR_NAMES)},
        'exact': {n: float((d[..., j] == 0).mean()) for j, n in enumerate(ATTR_NAMES)},
        'within1': {n: float((np.abs(d[..., j]) <= 1).mean()) for j, n in enumerate(ATTR_NAMES)},
    }
    print(f'{lab:>6} mean shift (got - requested):',
          {n[:5]: round(v, 2) for n, v in ATTR_DIAG[lab]['mean_shift'].items()})

LENGTHS = {k: {'mean': float(tok_len(v).mean()), 'median': float(np.median(tok_len(v))),
               'p90': float(np.percentile(tok_len(v), 90))} for k, v in SETS.items()}
print('token lengths:', {k: round(v['mean']) for k, v in LENGTHS.items()})

np.savez_compressed(OUT_DIR / 'tokens.npz', **SETS, attr_requested=ATTR_REQ,
                    A_idx=A_IDX[:N_DONE][keep], B_idx=B_IDX[:N_DONE][keep_B])

In [ ]:
# -------------------------
# Tokens -> MIDI -> CLaMP 2 embeddings for A, B, plain, simple, hier
# -------------------------

t_emb = time.time()
EMB = {}
set_names = list(SETS.keys())
pbar = tqdm(total=len(set_names) * N_ALL, desc='MIDI + CLaMP 2', unit='file')
for name in set_names:
    pbar.set_postfix(set=name)
    out_path = OUT_DIR / f'emb_{name}.npy'
    (SAMPLES_DIR / name).mkdir(parents=True, exist_ok=True)
    feats = np.zeros((N_ALL, EMB_DIM), dtype=np.float32)
    for i, row in enumerate(SETS[name]):
        feats[i], pth = embed_tokens(row, f'{name}_{i}')
        if i < int(CFG['N_MIDI_SAMPLES']):
            shutil.copy(pth, SAMPLES_DIR / name / f'{name}_{i:03d}.mid')
        pth.unlink()
        pbar.update(1)
    np.save(out_path, feats)
    EMB[name] = feats
pbar.close()
shutil.rmtree(MIDI_TMP, ignore_errors=True)
TIMINGS['embedding_s'] = time.time() - t_emb
print(f'embedded {len(set_names)} x {N_ALL} in {fmt_secs(TIMINGS["embedding_s"])}')
assert all(np.isfinite(v).all() for v in EMB.values())

# Free GPU memory for the tests (CLaMP 2 and the generators are no longer needed).
del EXTRACTOR, MODELS
import gc
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# -------------------------
# MMD-D tests
#   per split s: A and all Q_m share one permutation (index-aligned), B has its own.
#   comparisons: plain vs B, simple vs B, hier vs B, A vs B (control)
#   cross-kernel control: each model's frozen kernel on the control pair's test halves
# -------------------------

E = {k: torch.from_numpy(v).to(DEVICE) for k, v in EMB.items()}
n = N_ALL
n_te = int(n * CFG['FRAC_TEST'])
n_va = max(4, int(round(n * CFG['FRAC_VAL'])))
n_fi = n - n_te - n_va
assert n_fi >= 4, f'too few samples ({n}) for a fit/val/test split'
PAIRS = MODEL_LABELS + ['A']   # X side; Y side is always B
print(f'n per side {n}: target fit {n_fi}, val {n_va}, test {n_te} (split by blocks of {CFG["SPLIT_BLOCK_WIDTH"]} memmap indices; parts trimmed to equal size per side)')


def block_parts(mm_idx, rng):
    # Assign whole split blocks (SPLIT_BLOCK_WIDTH consecutive memmap indices) to test, then val, then fit,
    # in random block order, until each part reaches its target size. Rows are shuffled within parts.
    blk = mm_idx // int(CFG['SPLIT_BLOCK_WIDTH'])
    order = rng.permutation(np.unique(blk))
    te, va, fi = [], [], []
    n_te_got = n_va_got = 0
    for b in order:
        rows = np.flatnonzero(blk == b)
        if n_te_got < n_te:
            te.append(rows); n_te_got += len(rows)
        elif n_va_got < n_va:
            va.append(rows); n_va_got += len(rows)
        else:
            fi.append(rows)
    cat = lambda xs: rng.permutation(np.concatenate(xs)) if xs else np.zeros(0, dtype=np.int64)
    return cat(te), cat(va), cat(fi)


def equalize(a, b, rng):
    # The estimators need equal sizes per side; trim the larger part at random.
    m = min(len(a), len(b))
    return rng.permutation(a)[:m], rng.permutation(b)[:m]


def to_dev(x):
    return torch.as_tensor(x, dtype=torch.long, device=DEVICE)


RESULTS = {'splits': []}


def write_results(final: bool = False):
    payload = {
        'mode': 'dry' if DRY_RUN else 'full', 'final': final, 'config': CFG,
        'sampling': {'temperature': TEMPERATURE, 'top_p': TOP_P, 'start_token': 'Bar'},
        'N_requested': N, 'N_generated': N_DONE, 'N_ALL': N_ALL, 'split_sizes': [n_fi, n_va, n_te],
        'dropped': DROPPED, 'kv_check': KV_CHECK, 'attributes': ATTR_DIAG, 'lengths': LENGTHS,
        'timings': TIMINGS, 'elapsed_s': elapsed(), 'emb_dim': EMB_DIM, **RESULTS,
    }
    (OUT_DIR / 'results.json').write_text(json.dumps(payload, indent=1), encoding='utf-8')


t_tests = time.time()
pbar = tqdm(total=int(CFG['N_SPLITS']) * (len(PAIRS) + len(MODEL_LABELS) + 1), desc='MMD tests', unit='test')
stopped_early = False
for s in range(int(CFG['N_SPLITS'])):
    rng_s = np.random.default_rng(SEED * 7919 + 17 * s + 3)
    pX = block_parts(MM_A, rng_s)
    pB = block_parts(MM_B, rng_s)
    (te_X, te_B), (va_X, va_B), (fi_X, fi_B) = [equalize(a, b, rng_s) for a, b in zip(pX, pB)]
    sizes = {'fit': len(fi_X), 'val': len(va_X), 'test': len(te_X)}
    te_np_X, te_np_B = te_X, te_B
    te_X, va_X, fi_X, te_B, va_B, fi_B = map(to_dev, (te_X, va_X, fi_X, te_B, va_B, fi_B))
    split = {'split': s, 'sizes': sizes, 'tests': {}, 'cross_kernel_control': {}, 'shared': {}}
    kernels = {}
    for pi, xname in enumerate(PAIRS):
        if elapsed() > CFG['DEADLINE_S']:
            stopped_early = True
            break
        X, Y = E[xname], E['B']
        std = mg.Standardizer(torch.cat([X[fi_X], Y[fi_B]], 0))
        kernel, info = mg.train_deep_kernel(
            std(X[fi_X]), std(Y[fi_B]), std(X[va_X]), std(Y[va_B]),
            width=int(CFG['PHI_WIDTH']), lr=float(CFG['LR']), steps=int(CFG['STEPS']),
            val_every=int(CFG['VAL_EVERY']), seed=SEED * 100_003 + 101 * s + pi,
            early_stop=bool(CFG['EARLY_STOP']), patience=int(CFG['PATIENCE']),
        )
        g = torch.Generator(device=DEVICE).manual_seed(SEED * 31 + 1009 * s + pi)
        res = mg.permutation_test(kernel, std(X[te_X]), std(Y[te_B]), n_perm=int(CFG['N_PERM']), generator=g)
        res['reject'] = bool(res['p'] <= CFG['ALPHA'])
        split['tests'][xname] = {**res, **info}
        kernels[xname] = (kernel, std)
        pbar.update(1)
        pbar.set_postfix(split=s, last=f'{xname} J={res["J"]:.3f} p={res["p"]:.3f}')
    for mi, mname in enumerate(MODEL_LABELS):
        if mname not in kernels or elapsed() > CFG['DEADLINE_S']:
            continue
        kernel, std = kernels[mname]
        g = torch.Generator(device=DEVICE).manual_seed(SEED * 37 + 2003 * s + mi)
        res = mg.permutation_test(kernel, std(E['A'][te_X]), std(E['B'][te_B]), n_perm=int(CFG['N_PERM']), generator=g)
        res['reject'] = bool(res['p'] <= CFG['ALPHA'])
        split['cross_kernel_control'][mname] = res
        pbar.update(1)

    # Shared kernel + relative similarity: one kernel trained on B (fit) vs a pooled generated sample
    # (each fit index drawn from one of the three models at random), then for every pair (m1, m2):
    # D = MMD^2(Q_m1, B) - MMD^2(Q_m2, B) on the test halves, with an asymptotic z-test. The B-B term cancels.
    if elapsed() <= CFG['DEADLINE_S']:
        pick = rng_s.integers(0, len(MODEL_LABELS), size=n)
        pooled = torch.stack([E[m] for m in MODEL_LABELS], 0)[to_dev(pick), torch.arange(n, device=DEVICE)]
        Yb = E['B']
        std = mg.Standardizer(torch.cat([Yb[fi_B], pooled[fi_X]], 0))
        kernel, info = mg.train_deep_kernel(
            std(Yb[fi_B]), std(pooled[fi_X]), std(Yb[va_B]), std(pooled[va_X]),
            width=int(CFG['PHI_WIDTH']), lr=float(CFG['LR']), steps=int(CFG['STEPS']),
            val_every=int(CFG['VAL_EVERY']), seed=SEED * 100_003 + 101 * s + 97,
            early_stop=bool(CFG['EARLY_STOP']), patience=int(CFG['PATIENCE']),
        )
        Qte = {lab: std(E[lab][te_X]) for lab in GEN_SPECS}
        bsets = {str(w): (MM_A[te_np_X] // w, MM_B[te_np_B] // w) for w in CFG['BLOCK_WIDTHS']}
        rel = mg.relative_similarity(kernel, Qte, std(Yb[te_B]), REL_PAIRS, block_sets=bsets)
        # Robustness: the same relative test with a fixed Gaussian kernel (no training, median bandwidth).
        fixed = mg.fixed_gaussian_kernel(std(torch.cat([Yb[fi_B], pooled[fi_X]], 0)), EMB_DIM, int(CFG['PHI_WIDTH']))
        rel_fixed = mg.relative_similarity(fixed, Qte, std(Yb[te_B]), REL_PAIRS, block_sets=bsets)
        ctrl = mg.permutation_test(kernel, std(E['A'][te_X]), std(Yb[te_B]), n_perm=int(CFG['N_PERM']),
                                   generator=torch.Generator(device=DEVICE).manual_seed(SEED * 41 + 3001 * s))
        ctrl['reject'] = bool(ctrl['p'] <= CFG['ALPHA'])
        split['shared'] = {'train': info, **rel, 'control_A_vs_B': ctrl, 'fixed_kernel': rel_fixed}
        pbar.update(1)
        pbar.set_postfix(split=s, shared='done')
    RESULTS['splits'].append(split)
    TIMINGS['tests_s'] = time.time() - t_tests
    write_results(final=False)
    if stopped_early:
        print(f'deadline reached during split {s}; stopping tests')
        break
pbar.close()
print(f'tests done in {fmt_secs(time.time() - t_tests)} | splits completed: {len(RESULTS["splits"])}')

In [ ]:
# -------------------------
# Summary: ranking by mean held-out J (lower = harder to tell from real music with MMD-D)
# -------------------------

def agg(rows):
    if not rows:
        return None
    J = np.array([r['J'] for r in rows])
    M = np.array([r['mmd2'] for r in rows])
    out = {
        'n_splits': len(rows),
        'J_mean': float(J.mean()), 'J_sd': float(J.std(ddof=1)) if len(J) > 1 else 0.0,
        'mmd2_mean': float(M.mean()), 'mmd2_sd': float(M.std(ddof=1)) if len(M) > 1 else 0.0,
        'rejections': int(sum(r['reject'] for r in rows)),
        'p_median': float(np.median([r['p'] for r in rows])),
    }
    if 'fit_J' in rows[0]:
        out.update(fit_J_mean=float(np.mean([r['fit_J'] for r in rows])),
                   val_J_mean=float(np.mean([r['val_J'] for r in rows])),
                   best_step_mean=float(np.mean([r['best_step'] for r in rows])),
                   steps_run_mean=float(np.mean([r['steps_run'] for r in rows])))
    return out


SUMMARY = {'tests': {}, 'cross_kernel_control': {}}
for name in PAIRS:
    SUMMARY['tests'][name] = agg([sp['tests'][name] for sp in RESULTS['splits'] if name in sp['tests']])
for name in MODEL_LABELS:
    SUMMARY['cross_kernel_control'][name] = agg(
        [sp['cross_kernel_control'][name] for sp in RESULTS['splits'] if name in sp['cross_kernel_control']])


def fails(a):
    # 3 or more rejections in 10 (30%) counts as a failed control; needs at least 3 splits to judge.
    return a is not None and a['n_splits'] >= 3 and a['rejections'] / a['n_splits'] >= 0.3


ctrl = SUMMARY['tests'].get('A')
SUMMARY['control_failed'] = bool(fails(ctrl))
SUMMARY['cross_kernel_control_failed'] = {m: bool(fails(SUMMARY['cross_kernel_control'][m])) for m in MODEL_LABELS}
ranked = sorted([m for m in MODEL_LABELS if SUMMARY['tests'][m]], key=lambda m: SUMMARY['tests'][m]['J_mean'])
SUMMARY['ranking_low_J_first'] = ranked

# Shared kernel: absolute MMD^2 / J per set on one scale, and relative-similarity pairs.
sh = [sp['shared'] for sp in RESULTS['splits'] if sp.get('shared')]
SUMMARY['shared'] = {'n_splits': len(sh), 'absolute': {}, 'pairs': {}}
for lab in GEN_SPECS:
    if sh:
        SUMMARY['shared']['absolute'][lab] = {
            'mmd2_mean': float(np.mean([x['absolute'][lab]['mmd2'] for x in sh])),
            'J_mean': float(np.mean([x['absolute'][lab]['J'] for x in sh])),
        }
def agg_pairs(rel_list):
    outp = {}
    for a, b in REL_PAIRS:
        key = f'{a}-{b}'
        rows = [x['pairs'][key] for x in rel_list]
        if not rows:
            continue
        zc = [r['z_cluster'] for r in rows]
        outp[key] = {
            'D_mean': float(np.mean([r['D'] for r in rows])),
            'z_naive_mean': float(np.mean([r['z_naive'] for r in rows])),
            'z_cluster_mean': float(np.mean(zc)),
            'z_cluster_sd': float(np.std(zc, ddof=1)) if len(zc) > 1 else 0.0,
            'design_effect_mean': float(np.mean([r['design_effect'] for r in rows])),
            'p_cluster_median': float(np.median([r['p_cluster'] for r in rows])),
            'rejections_cluster': int(sum(r['p_cluster'] <= CFG['ALPHA'] for r in rows)),
            'rejections_naive': int(sum(r['p_naive'] <= CFG['ALPHA'] for r in rows)),
            'rejections_bonferroni': int(sum(r['p_cluster'] <= CFG['ALPHA'] / 3 for r in rows)),
            'design_effect_by_width': {str(w): float(np.mean([r[f'design_effect_{w}'] for r in rows])) for w in CFG['BLOCK_WIDTHS']},
            'first_closer_splits': int(sum(r['D'] < 0 for r in rows)),
            'n_splits': len(rows),
        }
    return outp


SUMMARY['shared']['pairs'] = agg_pairs(sh)
SUMMARY['shared']['pairs_fixed_kernel'] = agg_pairs([x['fixed_kernel'] for x in sh])
calib = SUMMARY['shared']['pairs'].get('plain-plain2')
SUMMARY['shared']['calibration_failed'] = bool(fails({'n_splits': calib['n_splits'], 'rejections': calib['rejections_cluster']})) if calib else None
if sh:
    c = [x['control_A_vs_B'] for x in sh]
    SUMMARY['shared']['control_A_vs_B'] = {'rejections': int(sum(r['reject'] for r in c)), 'n_splits': len(c),
                                           'J_mean': float(np.mean([r['J'] for r in c]))}
RESULTS['summary'] = SUMMARY
write_results(final=True)

lines = [f'# MMD-D two-sample test ({"dry run" if DRY_RUN else "full run"})', '',
         f'N per side: {N_ALL} (generated {N_DONE}, dropped {DROPPED}); splits: {len(RESULTS["splits"])}; '
         f'fit/val/test per side: {n_fi}/{n_va}/{n_te}; permutations: {CFG["N_PERM"]}; alpha {CFG["ALPHA"]}.', '',
         '| vs B | held-out J (mean ± sd) | held-out MMD² | rejections | median p | fit J | val J | cross-kernel control rejections |',
         '|---|---|---|---|---|---|---|---|']
for name in ranked + ['A']:
    a = SUMMARY['tests'][name]
    if a is None:
        continue
    c = SUMMARY['cross_kernel_control'].get(name)
    cc = f'{c["rejections"]}/{c["n_splits"]}' if c else '(control)'
    lines.append(f'| {name} | {a["J_mean"]:.4f} ± {a["J_sd"]:.4f} | {a["mmd2_mean"]:.5f} | {a["rejections"]}/{a["n_splits"]} | '
                 f'{a["p_median"]:.3f} | {a["fit_J_mean"]:.3f} | {a["val_J_mean"]:.3f} | {cc} |')
lines += ['', f'Ranking (lower held-out J = harder to tell from real music under its own kernel): {" < ".join(ranked)}',
          f'Control failed: {SUMMARY["control_failed"]}; cross-kernel control failed: {SUMMARY["cross_kernel_control_failed"]}']

shs = SUMMARY['shared']
if shs['n_splits']:
    lines += ['', '## Relative similarity (headline comparison)', '',
              'Per split: D = MMD²(first, B) − MMD²(second, B) on the test halves; D < 0 means the first set is closer to real music.',
              'z_cluster: two-way cluster-robust SE over piece blocks, the more conservative of block widths '
              + ' and '.join(str(w) for w in CFG['BLOCK_WIDTHS']) + ' (decide on this); z_naive assumes independent rows.',
              'design effect = clustered / naive variance (per width). Bonferroni: p ≤ α/3 for the three model pairs. '
              'plain vs plain2 is the calibration (true D = 0).', '']
    for title, key in [('Learned shared kernel', 'pairs'), ('Fixed Gaussian kernel (robustness)', 'pairs_fixed_kernel')]:
        lines += [f'**{title}**', '',
                  '| pair | mean D | mean z_cluster (± sd) | mean z_naive | design effect by width | rejections (cluster) | rejections (Bonferroni) | median p (cluster) | first closer in |',
                  '|---|---|---|---|---|---|---|---|---|']
        for pk, r in shs[key].items():
            lines.append(f'| {pk} | {r["D_mean"]:+.5f} | {r["z_cluster_mean"]:+.2f} ± {r["z_cluster_sd"]:.2f} | {r["z_naive_mean"]:+.2f} | '
                         + ' / '.join(f'{w}: {v:.1f}' for w, v in r['design_effect_by_width'].items()) + ' | '
                         f'{r["rejections_cluster"]}/{r["n_splits"]} | {r["rejections_bonferroni"]}/{r["n_splits"]} | {r["p_cluster_median"]:.3g} | '
                         f'{r["first_closer_splits"]}/{r["n_splits"]} splits |')
        lines.append('')
    lines += ['', '| set vs B (shared kernel) | mean MMD² | mean J |', '|---|---|---|']
    for lab, r in shs['absolute'].items():
        lines.append(f'| {lab} | {r["mmd2_mean"]:.5f} | {r["J_mean"]:.3f} |')
    c = shs['control_A_vs_B']
    lines += ['', f'Calibration (plain vs plain2) failed: {shs["calibration_failed"]}; '
              f'shared-kernel control A vs B: {c["rejections"]}/{c["n_splits"]} rejections, mean J {c["J_mean"]:.3f}']

lines += ['', '## Attribute shift (recomputed bin minus requested bin)', '']
for m in GEN_SPECS:
    lines.append(f'- {m}: ' + ', '.join(f'{k} {v:+.2f}' for k, v in ATTR_DIAG[m]['mean_shift'].items()))
lines += ['', '## Token lengths (mean)', '', ', '.join(f'{k} {v["mean"]:.0f}' for k, v in LENGTHS.items()),
          '', f'Elapsed: {fmt_secs(elapsed())}']
(OUT_DIR / 'summary.md').write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))
print('\nwrote', OUT_DIR / 'results.json', 'and', OUT_DIR / 'summary.md')